# Task 1: train S3 (BERT) and S4 (RoBERTa) on Colab

Runtime → Change runtime type → **T4 GPU**. Run all cells in order.

1. Mount Drive (weights + results are saved there so they survive a disconnect).
2. Clone the repo branch and install pinned deps.
3. Re-create the exact Task 1 splits (checked against the local fingerprint).
4. Train S4 then S3 (max_length 256, fp16), evaluate each.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
OUT = '/content/drive/MyDrive/steam-review-translator-task1'
os.makedirs(OUT, exist_ok=True)
os.environ['ARTIFACTS_DIR'] = OUT  # models -> OUT/models/task1, results -> OUT/runs/task1
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
BRANCH = 'task1/data-and-eval'
!git clone -q --branch $BRANCH https://github.com/mahesa005/steam-review-translator.git /content/srt
%cd /content/srt
!pip install -q -e '.[task1]' pandas==3.1.1 numpy==2.5.5 pyarrow==26.0.1 py3langid==0.4.0 scikit-learn==1.9.1

In [ ]:
# Same data as the local machine: download, prepare, and verify the fingerprint.
!make task1-download PYTHON=python 2>&1 | tail -2
!cd experiments/task1_sentiment && python prepare_metacritic.py 2>&1 | grep -v Warning | tail -3 && python prepare_steam.py --no-names 2>&1 | grep -v Warning | tail -2
out = !python experiments/task1_sentiment/fingerprint.py
print('\n'.join(out))
EXPECTED = {'metacritic.parquet': '0a8b9967d1189456', 'steam.parquet': '661e2fd1d614ab76'}
for name, h in EXPECTED.items():
    assert any(name in l and h in l for l in out), f'{name} differs from the local split: STOP and tell Farrel'
print('fingerprints match')

In [ ]:
# S4: RoBERTa-base (main system)
!python experiments/task1_sentiment/train_transformer.py --model roberta-base --max-length 256 --fp16 --batch-size 32 --lr 2e-5 2>&1 | grep -vE 'Warning|warn'
!python experiments/task1_sentiment/evaluate.py --system transformer --name S4-roberta-len256 --model $OUT/models/task1/roberta-base_len256 --batch-size 64 2>&1 | grep -E '===|metacritic_test|steam |->'

In [ ]:
# S3: BERT-base
!python experiments/task1_sentiment/train_transformer.py --model bert-base-uncased --max-length 256 --fp16 --batch-size 32 --lr 2e-5 2>&1 | grep -vE 'Warning|warn'
!python experiments/task1_sentiment/evaluate.py --system transformer --name S3-bert-len256 --model $OUT/models/task1/bert-base-uncased_len256 --batch-size 64 2>&1 | grep -E '===|metacritic_test|steam |->'

In [ ]:
# Summary of every finished run (copy this output back)
import json, glob
for f in sorted(glob.glob(f'{OUT}/models/task1/*/train_log.json')):
    print(f.split('/')[-2], 'best val macro-F1', round(json.load(open(f))['best_val_macro_f1'], 4))
for f in sorted(glob.glob(f'{OUT}/runs/task1/*/metrics.json')):
    m = json.load(open(f))
    for k in ('metacritic_test', 'steam'):
        r, d = m[k]['review'], m[k]['distribution']
        print(m['system'], k, 'acc', round(r['accuracy'], 3), 'macroF1', round(r['macro_f1'], 3), 'TVD', round(d['cc']['tvd'], 3), 'ACC-TVD', round(d['acc']['tvd'], 3))

Optional: max_length 512 (about 2x slower). Pick the length by **validation** macro-F1 in `train_log.json`, then report test only for the chosen one.

In [ ]:
# !python experiments/task1_sentiment/train_transformer.py --model roberta-base --max-length 512 --fp16 --batch-size 16 2>&1 | grep -vE 'Warning|warn'